##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

#### 1) Setup

In [2]:
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

#### 2) Load CIFAR-10

In [4]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

# Keep labels as integers for SparseCategoricalCrossentropy
y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

# Convert to float32
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_test :", x_test.shape,  "y_test :", y_test.shape)

x_train: (50000, 32, 32, 3) y_train: (50000,)
x_test : (10000, 32, 32, 3) y_test : (10000,)


#### 3) Data Augmentation

In [5]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ],
    name="augmentation"
)

#### 4) Build MobileNetV2 Transfer Learning Model

In [6]:
# Build MobileNetV2 backbone (pretrained)
mobilenet_base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

# Freeze first (feature extractor)
mobilenet_base.trainable = False

# Full model (preprocess inside model)
mobilenet_model = keras.Sequential(
    [
        layers.Input(shape=(32, 32, 3)),
        data_augmentation,
        layers.Resizing(224, 224, interpolation="bilinear"),
        layers.Lambda(mobilenet_preprocess),   # IMPORTANT
        mobilenet_base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10)                       # logits
    ],
    name="cifar10_mobilenetv2"
)

mobilenet_model.summary()

Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

#### 5) Compile + Train (Frozen Backbone)

In [8]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

t0 = time.perf_counter()
history_mn_frozen = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
t_frozen = time.perf_counter() - t0

test_loss_mn_frozen, test_acc_mn_frozen = mobilenet_model.evaluate(x_test, y_test, verbose=0)

print("\nMobileNetV2 (frozen) test accuracy:", test_acc_mn_frozen)
print("MobileNetV2 (frozen) test loss    :", test_loss_mn_frozen)
print("Frozen training time (sec):", t_frozen)

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 246s 347ms/step - accuracy: 0.7591 - loss: 0.6967 - val_accuracy: 0.8272 - val_loss: 0.5112
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 244s 346ms/step - accuracy: 0.7664 - loss: 0.6695 - val_accuracy: 0.8320 - val_loss: 0.5114
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 244s 347ms/step - accuracy: 0.7694 - loss: 0.6620 - val_accuracy: 0.8360 - val_loss: 0.4933

MobileNetV2 (frozen) test accuracy: 0.8205999732017517
MobileNetV2 (frozen) test loss    : 0.5294266939163208
Frozen training time (sec): 734.6390579000581


#### 6) Fine-tune MobileNetV2 (Unfreeze last layers)

In [10]:
# Unfreeze backbone
mobilenet_base.trainable = True

# Freeze most layers, unfreeze last N layers
N = 30
for layer in mobilenet_base.layers[:-N]:
    layer.trainable = False

# Common trick: keep BatchNorm frozen
for layer in mobilenet_base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print("Trainable layers in backbone:",
      sum(int(l.trainable) for l in mobilenet_base.layers), "/", len(mobilenet_base.layers))

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

t0 = time.perf_counter()
history_mn_ft = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
t_ft = time.perf_counter() - t0

test_loss_mn_ft, test_acc_mn_ft = mobilenet_model.evaluate(x_test, y_test, verbose=0)

print("\nMobileNetV2 (fine-tuned) test accuracy:", test_acc_mn_ft)
print("MobileNetV2 (fine-tuned) test loss    :", test_loss_mn_ft)
print("Fine-tune training time (sec):", t_ft)

Trainable layers in backbone: 19 / 154
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 275s 386ms/step - accuracy: 0.7944 - loss: 0.5905 - val_accuracy: 0.8558 - val_loss: 0.4232
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 270s 383ms/step - accuracy: 0.8090 - loss: 0.5443 - val_accuracy: 0.8604 - val_loss: 0.4109
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 268s 381ms/step - accuracy: 0.8210 - loss: 0.5109 - val_accuracy: 0.8676 - val_loss: 0.3810

MobileNetV2 (fine-tuned) test accuracy: 0.858299970626831
MobileNetV2 (fine-tuned) test loss    : 0.41435369849205017
Fine-tune training time (sec): 813.20031320001


#### 7) Collect Results + Compare Models

In [12]:
# === Summary ===:
custom_cnn_test_acc = 0.8741999864578247
resnet_frozen_test_acc = 0.8741999864578247
resnet_finetuned_test_acc = 0.9161999821662903

results = {
    "Custom CNN test acc": custom_cnn_test_acc,
    "ResNet50V2 frozen test acc": resnet_frozen_test_acc,
    "ResNet50V2 fine-tuned test acc": resnet_finetuned_test_acc,
    "MobileNetV2 frozen test acc": float(test_acc_mn_frozen),
    "MobileNetV2 fine-tuned test acc": float(test_acc_mn_ft),
}

times = {
    "MobileNetV2 frozen train time (sec)": float(t_frozen),
    "MobileNetV2 fine-tune train time (sec)": float(t_ft),
}

print("=== Accuracy Comparison ===")
for k, v in results.items():
    print(f"{k:30s}: {v}")

print("\n=== Timing ===")
for k, v in times.items():
    print(f"{k:35s}: {v:.2f}")

=== Accuracy Comparison ===
Custom CNN test acc           : 0.8741999864578247
ResNet50V2 frozen test acc    : 0.8741999864578247
ResNet50V2 fine-tuned test acc: 0.9161999821662903
MobileNetV2 frozen test acc   : 0.8205999732017517
MobileNetV2 fine-tuned test acc: 0.858299970626831

=== Timing ===
MobileNetV2 frozen train time (sec): 734.64
MobileNetV2 fine-tune train time (sec): 813.20


In [13]:
best_model = max(results, key=results.get)
print("\nBest accuracy:", best_model, "=", results[best_model])


Best accuracy: ResNet50V2 fine-tuned test acc = 0.9161999821662903


#### 1) Which model achieved the highest accuracy?

ResNet50V2 fine-tuned achieved the highest accuracy (≈ 91.6%).

> The fine-tuned ResNet50V2 model achieved the best performance because its deep residual architecture allows learning complex hierarchical image features, and fine-tuning enables adaptation of pretrained ImageNet features to CIFAR-10.


#### 2) Which model trained faster?

- MobileNet uses depthwise separable convolutions

- drastically fewer computations

- designed for mobile / efficiency

> MobileNetV2 trained faster because it is designed as a lightweight architecture with fewer parameters and computational operations compared to ResNet50V2.


#### 3) How does the architecture explain the differences?

Custom CNN

- trained from scratch

- no pretrained knowledge

- fewer learned high-level features

- therefore lower accuracy


MobileNetV2

- lightweight architecture

- uses depthwise separable convolutions

- fewer parameters → faster training

- but slightly reduced representational power


ResNet50V2

- very deep network

- residual connections solve vanishing gradients

- can learn complex image representations

- fine-tuning adapts pretrained features → highest accuracy